# PhenoPrompt — Stage 3: Prompt-Based Phenotype Query Interface

**Goal:** Turn the phenotype clusters discovered in Stage 2 into a **queryable phenotype
space**. A user types a free-text clinical prompt — e.g. *"show me clusters related to
type 2 diabetes with renal complications"* — and receives a ranked list of **interpretable
phenotype reports**, one per matched cluster.

This is the novel contribution of PhenoPrompt: rather than hard-coding one algorithm per
disease, *any* condition can be queried interactively against a single index that was built
**without** disease-specific specification.

Retrieval is **entity-augmented** (inspired by the CLEAR pipeline, López et al., *npj Digital
Medicine* 2025): clusters are scored by their medkit-extracted **phenotype mix**, not by dense
note embeddings. An optional dense term gives a hybrid score. The LLM only *synthesises* the
final report from retrieved, grounded evidence — it never drives the retrieval.

---

### Pipeline overview

```
stage1_outputs/                      stage2_outputs/
  entity_count_matrix.csv              phenotype_profiles.json   <- cluster phenotype mixes
  entity_mentions.csv (provenance)     cluster_assignments.csv
  note_ids.npy                         svd_embeddings.npy
  notes.csv  (or reload from HF)       umap_2d_coords.csv
         |                                     |
         +------------------+------------------+
                            v
                  PhenoSpace  (queryable index)
                            |
   free-text prompt --> extract query entities (synonym match / medkit)
                            |
                            v
         entity-augmented + dense hybrid cluster retrieval
                            |
                            v
   per-cluster:  phenotype-mix table . grounded note fragments + provenance
                            |
                            v
        LLM synthesis  (Anthropic / OpenAI / local / template fallback)
                            |
                            v
   Level 1 per-query reports . Level 2 population map . Level 3 novel-phenotype flags
```

> **Always runs.** Every external dependency degrades gracefully: no API key -> deterministic
> template reports; no `plotly` -> matplotlib only; no structured ICD codes -> the under-coding
> analysis is skipped with a clear message. The retrieval core needs nothing beyond NumPy/pandas.


## 0. Install dependencies

In [ ]:
# Run once, then restart the kernel if Colab asks.
import subprocess, sys

packages = [
    "pandas", "numpy", "scikit-learn", "scipy",
    "matplotlib", "seaborn", "plotly",
    "datasets",          # to recover note texts if notes.csv is absent
    # -- LLM backends (all optional - install only what you'll use) --
    "anthropic",         # backend = "anthropic"
    "openai",            # backend = "openai"
    # "transformers", "torch", "accelerate",   # backend = "local"
    # "medkit-lib",      # only if you want medkit NER on the query string
]
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    except Exception as e:
        print(f"  (skipped {pkg}: {e})")
print("Dependencies ready.")

## 1. Imports & configuration

In [ ]:
import os, re, json, textwrap, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# -- Directories (produced by Stage 1 / Stage 2) ------------------------------
from google.colab import drive
drive.mount("/content/drive")
BASE       = Path("/content/drive/MyDrive/phenoprompt")
STAGE1_DIR = BASE / "stage1_outputs"
STAGE2_DIR = BASE / "stage2_outputs"
STAGE3_DIR = BASE / "stage3_outputs"; STAGE3_DIR.mkdir(parents=True, exist_ok=True)

# -- Retrieval hyperparameters ------------------------------------------------
TOP_K     = 3       # clusters returned per query
ALPHA     = 0.7     # hybrid weight: ALPHA*entity_score + (1-ALPHA)*dense_cosine
MIN_SCORE = 1e-6    # drop clusters with no entity overlap

# -- LLM backend for report synthesis -----------------------------------------
# "fallback"  -> deterministic template (no API key, always works)
# "anthropic" -> set ANTHROPIC_API_KEY ; pick a model you have access to
# "openai"    -> set OPENAI_API_KEY
# "local"     -> small HF model (needs transformers+torch); good for offline Colab
LLM_BACKEND     = "fallback"
ANTHROPIC_MODEL = "claude-3-5-sonnet-latest"   # change to a model you can call
OPENAI_MODEL    = "gpt-4o-mini"
LOCAL_MODEL     = "google/flan-t5-base"

# In Colab you can do:  os.environ["ANTHROPIC_API_KEY"] = "sk-..."
# or, more safely:      from google.colab import userdata
#                       os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

print(f"TOP_K={TOP_K}  ALPHA={ALPHA}  LLM_BACKEND={LLM_BACKEND!r}")

## 2. Recover note texts for grounding

The clinical narrative is grounded on real note fragments, so we need the raw text aligned
with `note_id`. Stage 1 can save these to `stage1_outputs/notes.csv`; if that file is absent
we reload the **same** corpus from HuggingFace using Stage 1's seed/size so the IDs line up.

> If you changed `N_NOTES` or `RANDOM_SEED` in Stage 1, set them identically here.

In [ ]:
NOTES_CSV = STAGE1_DIR / "notes.csv"
note_texts = {}

if NOTES_CSV.exists():
    _df = pd.read_csv(NOTES_CSV, dtype={"idx": str})
    note_texts = dict(zip(_df["idx"], _df["note"]))
    print(f"Loaded {len(note_texts):,} note texts from {NOTES_CSV}")
else:
    print("notes.csv not found - reloading AGBonnet/augmented-clinical-notes from HuggingFace.")
    try:
        from datasets import load_dataset
        N_NOTES, RANDOM_SEED = 500, 42        # <- MUST match Stage 1
        ds = load_dataset("AGBonnet/augmented-clinical-notes", split="train")
        if N_NOTES is not None:
            ds = ds.shuffle(seed=RANDOM_SEED).select(range(N_NOTES))
        _df = ds.to_pandas()
        note_texts = {str(i): t for i, t in zip(_df["idx"], _df["note"])}
        print(f"Recovered {len(note_texts):,} note texts from HuggingFace.")
    except Exception as e:
        print(f"  Could not reload dataset ({e}).")
        print("  Narratives still work but fragments may be empty.")
        print("  Tip: add a cell to Stage 1 that saves df[['idx','note']] to notes.csv.")

## 3. Build the queryable PhenoSpace index

`PhenoSpace` loads every Stage 1/2 artifact and precomputes, per cluster, an entity
**prevalence vector** over the full vocabulary (the dense hybrid term) and exposes the
phenotype mix (the entity-augmented term).

In [ ]:
class PhenoSpace:
    """A queryable index over the phenotype clusters discovered in Stage 2."""

    def __init__(self, stage1_dir, stage2_dir, note_texts=None):
        s1, s2 = Path(stage1_dir), Path(stage2_dir)
        self.profiles = json.loads((s2 / "phenotype_profiles.json").read_text())
        self.assign   = pd.read_csv(s2 / "cluster_assignments.csv", dtype={"note_id": str})
        self.count    = pd.read_csv(s1 / "entity_count_matrix.csv", index_col=0)
        self.count.index = self.count.index.astype(str)
        self.vocab    = list(self.count.columns)
        self.mentions = pd.read_csv(s1 / "entity_mentions.csv", dtype={"note_id": str})
        self.note_texts = note_texts or {}

        self.cluster_ids = sorted(int(c) for c in self.profiles)
        self.corpus_prev = (self.count > 0).mean()
        self.prev_vec    = {}
        for cl in self.cluster_ids:
            ids = self.assign.loc[self.assign.cluster == cl, "note_id"].values
            pv  = (self.count.reindex(ids) > 0).mean().reindex(self.vocab).fillna(0).values
            self.prev_vec[cl] = pv

    # -- query understanding: NL prompt -> weighted entities in vocab ----------
    def extract_query_entities(self, query, synonyms=None):
        q = query.lower()
        synonyms = synonyms or SYNONYMS
        hits = {}
        for e in self.vocab:                      # 1. direct surface-form match
            if e in q:
                hits[e] = max(hits.get(e, 0.0), 1.0)
        for syn, targets in synonyms.items():     # 2. synonym expansion
            if syn in q:
                for t in targets:
                    if t in self.vocab:
                        hits[t] = max(hits.get(t, 0.0), 0.9)
        for tok in re.findall(r"[a-z]+", q):      # 3. single-word fallback
            if len(tok) < 4:
                continue
            for e in self.vocab:
                if tok in e.split():
                    hits[e] = max(hits.get(e, 0.0), 0.7)
        return hits

    # -- entity-augmented + dense hybrid retrieval -----------------------------
    def retrieve(self, query, top_k=None, alpha=None, synonyms=None):
        top_k = TOP_K if top_k is None else top_k
        alpha = ALPHA if alpha is None else alpha
        qents = self.extract_query_entities(query, synonyms)
        if not qents:
            return [], qents
        vidx = {e: i for i, e in enumerate(self.vocab)}
        cp   = self.corpus_prev.reindex(self.vocab).fillna(0).values   # corpus prevalence
        qvec = np.array([qents.get(e, 0.0) for e in self.vocab], dtype=float)
        qn   = qvec / (np.linalg.norm(qvec) + 1e-8)

        ent_scores, cosines, rows = [], [], []
        for cl in self.cluster_ids:
            pv = self.prev_vec[cl]
            # entity-augmented score over the FULL vocab (not just the top-8 mix),
            # weighted by ENRICHMENT: prevalence x lift (cluster vs corpus). This
            # rewards clusters where the query entities are over-represented, so a
            # common entity like "diabetes" cannot pull in unrelated clusters.
            ent = 0.0
            for e, w in qents.items():
                if e in vidx:
                    i = vidx[e]
                    lift = pv[i] / (cp[i] + 1e-8)
                    ent += w * pv[i] * lift
            pvn = pv / (np.linalg.norm(pv) + 1e-8)
            cos = float(qn @ pvn)
            ent_scores.append(ent); cosines.append(cos)
            matched = [e for e in qents if e in vidx and pv[vidx[e]] > 0]
            rows.append(dict(cluster=cl, matched_entities=matched,
                             n_notes=self.profiles[str(cl)]["n_notes"]))
        ent_scores, cosines = np.array(ent_scores), np.array(cosines)
        def mm(a):                                # scale-free min-max across candidates
            rng = a.max() - a.min()
            return (a - a.min()) / rng if rng > 1e-12 else np.zeros_like(a)
        hybrid = alpha * mm(ent_scores) + (1 - alpha) * mm(cosines)
        for r, h, e, c in zip(rows, hybrid, ent_scores, cosines):
            r.update(hybrid=round(float(h), 4),
                     entity_score=round(float(e), 4), cosine=round(float(c), 4))
        rows = [r for r in rows if r["entity_score"] > MIN_SCORE]
        rows.sort(key=lambda r: r["hybrid"], reverse=True)
        return rows[:top_k], qents

    # -- grounding fragments + provenance for a matched cluster ----------------
    def fragments(self, cluster_id, query_entities, n=3):
        ids = set(self.assign.loc[self.assign.cluster == cluster_id, "note_id"])
        m = self.mentions[self.mentions.note_id.isin(ids)
                          & self.mentions.text.isin(list(query_entities))
                          & (self.mentions.assertion == "affirmed")]
        frags = []
        for nid in list(dict.fromkeys(m.note_id))[:n]:
            spans = m[m.note_id == nid][["text", "start", "end"]].to_dict("records")
            frags.append(dict(note_id=nid,
                              text=self.note_texts.get(nid, ""),
                              provenance=spans))
        return frags

    def phenotype_mix(self, cluster_id, top=10):
        return pd.DataFrame(self.profiles[str(cluster_id)]["top_entities"]).head(top)


# Curated synonym map: free-text phrasings -> vocabulary surface forms.
# Extend this for your corpus; it is the cheap, transparent alternative to medkit-on-query.
SYNONYMS = {
    "type 2 diabetes": ["diabetes"], "t2dm": ["diabetes"], "diabetic": ["diabetes"],
    "renal": ["chronic kidney disease", "renal failure", "kidney"],
    "kidney": ["chronic kidney disease", "renal failure"], "ckd": ["chronic kidney disease"],
    "hf": ["heart failure"], "chf": ["heart failure"], "cardiac failure": ["heart failure"],
    "fluid overload": ["edema"], "swelling": ["edema"], "sob": ["shortness of breath"],
    "breathless": ["shortness of breath"], "diuretic": ["furosemide"],
    "lung infection": ["pneumonia"], "chest infection": ["pneumonia"],
    "heart attack": ["myocardial infarction"], "high blood pressure": ["hypertension"],
}

ps = PhenoSpace(STAGE1_DIR, STAGE2_DIR, note_texts)
print(f"PhenoSpace ready: {len(ps.cluster_ids)} clusters, "
      f"{len(ps.vocab)} entities in vocabulary, {len(ps.note_texts):,} note texts.")

## 4. Query understanding - natural language to clinical entities

The default path matches the prompt against the cluster vocabulary using surface forms +
a curated synonym map. It is transparent and needs no models. For strict consistency with
Stage 1's extractor, the optional cell below runs the **same medkit NER** on the query.

In [ ]:
for demo_q in [
    "show me clusters related to type 2 diabetes with renal complications",
    "heart failure with fluid overload",
    "elderly patient breathless on a diuretic",
]:
    print(f"{demo_q!r}\n   -> {ps.extract_query_entities(demo_q)}\n")

In [ ]:
# OPTIONAL - medkit NER on the query string (consistent with Stage 1).
# Requires medkit-lib. If it fails, the synonym matcher above is used instead.
def medkit_query_entities(query):
    import importlib.util, sys as _sys
    site = [p for p in _sys.path if "site-packages" in p or "dist-packages" in p]
    cand = next((Path(d)/"medkit"/"text"/"ner"/"regexp_matcher.py"
                 for d in site if (Path(d)/"medkit"/"text"/"ner"/"regexp_matcher.py").exists()), None)
    spec = importlib.util.spec_from_file_location("medkit.text.ner.regexp_matcher", cand)
    mod  = importlib.util.module_from_spec(spec); _sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    # Reuse the SAME rule set you defined in Stage 1 (paste `all_rules` or import it),
    # run the matcher on the query text, then map matched entity ids/texts to the vocab.
    raise NotImplementedError("Paste your Stage 1 all_rules here to enable medkit-on-query.")

print("medkit-on-query available as medkit_query_entities() - wire in your Stage 1 rules to use it.")

## 5. Entity-augmented + hybrid cluster retrieval

`entity_score` is the CLEAR-style term: query-entity weight x cluster phenotype score.
`cosine` compares the query entity vector to each cluster's prevalence vector. Both are
min-max normalised across candidates and blended by `ALPHA`.

In [ ]:
ranked, qents = ps.retrieve("type 2 diabetes with renal complications")
print("Query entities:", qents, "\n")
if ranked:
    print(pd.DataFrame(ranked)[["cluster", "hybrid", "entity_score", "cosine",
                                "n_notes", "matched_entities"]].to_string(index=False))
else:
    print("No clusters matched. The query entities are not present in any cluster.")
    print("Try terms that appear in your corpus, or extend the SYNONYMS map.")

## 6. LLM synthesis layer

The LLM receives only the **phenotype mix** and **grounded note fragments**, and is asked for
(1) a short human-readable cluster label and (2) a 2-3 sentence narrative grounded strictly in
the supplied evidence. JSON out, with robust parsing and a deterministic template on any failure.

In [ ]:
def _build_prompt(mix_df, fragments, matched):
    mix_lines = "\n".join(
        f"- {r.entity} (in {r.prevalence_cluster:.0%} of cluster notes)"
        for r in mix_df.head(8).itertuples())
    frag_lines = "\n".join(
        f"- [{f['note_id']}] {f['text'][:300]}" for f in fragments) or "- (no fragments)"
    return f"""You are a clinical informatics assistant. Summarise ONE patient cluster.
Use ONLY the evidence below. Do not invent diagnoses or numbers.

Dominant entities (phenotype mix):
{mix_lines}

Representative note fragments:
{frag_lines}

The user's query matched on: {', '.join(matched) or 'overlapping entities'}.

Return STRICT JSON only:
{{"label": "<=8-word clinical name", "narrative": "2-3 grounded sentences"}}"""

def _template_report(mix_df, n_notes, matched):
    top = mix_df["entity"].head(4).tolist()
    label = " + ".join(t.title() for t in top[:2]) + " phenotype"
    narrative = (f"This cluster ({n_notes} notes) is dominated by {', '.join(top)}. "
                 f"The query matched on {', '.join(matched) or 'overlapping entities'}. "
                 f"Entity prevalences are listed in the phenotype-mix table.")
    return label, narrative

def _parse_json(text, mix_df, n_notes, matched):
    try:
        m = re.search(r"\{.*\}", text, re.S)
        obj = json.loads(m.group(0))
        return obj["label"].strip(), obj["narrative"].strip()
    except Exception:
        return _template_report(mix_df, n_notes, matched)

# backend clients (lazy init, cached) -----------------------------------------
_clients = {}
def _llm_complete(prompt):
    if LLM_BACKEND == "anthropic":
        if "a" not in _clients:
            import anthropic; _clients["a"] = anthropic.Anthropic()
        r = _clients["a"].messages.create(model=ANTHROPIC_MODEL, max_tokens=300,
                                          messages=[{"role": "user", "content": prompt}])
        return r.content[0].text
    if LLM_BACKEND == "openai":
        if "o" not in _clients:
            from openai import OpenAI; _clients["o"] = OpenAI()
        r = _clients["o"].chat.completions.create(model=OPENAI_MODEL, max_tokens=300,
                                                  messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if LLM_BACKEND == "local":
        if "l" not in _clients:
            from transformers import pipeline
            _clients["l"] = pipeline("text2text-generation", model=LOCAL_MODEL)
        return _clients["l"](prompt, max_new_tokens=200)[0]["generated_text"]
    raise RuntimeError("fallback")

def synthesize(mix_df, n_notes, matched, fragments):
    if LLM_BACKEND == "fallback":
        return _template_report(mix_df, n_notes, matched)
    try:
        return _parse_json(_llm_complete(_build_prompt(mix_df, fragments, matched)),
                           mix_df, n_notes, matched)
    except Exception as e:
        print(f"  (LLM backend '{LLM_BACKEND}' failed: {e} - using template)")
        return _template_report(mix_df, n_notes, matched)

print(f"Synthesis layer ready (backend={LLM_BACKEND!r}).")

## 7. The `query()` interface - Level 1 per-query reports

Ties retrieval, grounding and synthesis together. Each report carries: cluster label & size,
hybrid score, phenotype-mix table, grounded narrative, and provenance-bearing fragments.

In [ ]:
def query(prompt, top_k=None, verbose=True):
    ranked, qents = ps.retrieve(prompt, top_k=top_k)
    reports = []
    for r in ranked:
        mix   = ps.phenotype_mix(r["cluster"])
        frags = ps.fragments(r["cluster"], qents, n=2)
        label, narrative = synthesize(mix, r["n_notes"], r["matched_entities"], frags)
        reports.append(dict(cluster=r["cluster"], label=label, narrative=narrative,
                            hybrid_score=r["hybrid"], n_notes=r["n_notes"],
                            matched_entities=r["matched_entities"],
                            phenotype_mix=mix, fragments=frags))
    if verbose:
        _print_reports(prompt, qents, reports)
    return reports, qents

def _print_reports(prompt, qents, reports):
    print("=" * 78)
    print(f"QUERY: {prompt}")
    print(f"Extracted entities: {qents}")
    print("=" * 78)
    if not reports:
        print("No matching clusters. Try different terms or extend SYNONYMS."); return
    for i, rep in enumerate(reports, 1):
        print(f"\n[{i}] Cluster {rep['cluster']}  -  {rep['label']}")
        print(f"    {rep['n_notes']} patients  |  hybrid score {rep['hybrid_score']:.3f}"
              f"  |  matched: {', '.join(rep['matched_entities']) or '-'}")
        print(f"    {textwrap.fill(rep['narrative'], 92, subsequent_indent='    ')}")
        print("    Phenotype mix (top 6):")
        for r in rep["phenotype_mix"].head(6).itertuples():
            corpus_p = float(ps.corpus_prev.get(r.entity, 0.0))
            print(f"        {r.entity:28s} score={r.score:7.3f}  "
                  f"cluster={r.prevalence_cluster:.0%}  corpus={corpus_p:.0%}")
        if rep["fragments"] and rep["fragments"][0]["provenance"]:
            f = rep["fragments"][0]
            spans = [(s["text"], s["start"], s["end"]) for s in f["provenance"][:3]]
            print(f"    Provenance e.g. note {f['note_id']}: {spans}")

# -- Run a few example queries (Level 1 output) -------------------------------
EXAMPLE_QUERIES = [
    "show me clusters related to type 2 diabetes with renal complications",
    "heart failure with fluid overload",
    "respiratory infection with fever and cough",
]
all_results = {q: query(q)[0] for q in EXAMPLE_QUERIES}

## 8. Level 2 - population map with query highlight

The permanent embedding map from Stage 2, recoloured so the clusters matched by a query light
up against a greyed-out background. This is the researcher-facing, navigable phenotype atlas.

In [ ]:
umap_df = pd.read_csv(STAGE2_DIR / "umap_2d_coords.csv", dtype={"note_id": str})

def plot_query_map(prompt, top_k=None):
    ranked, _ = ps.retrieve(prompt, top_k=top_k)
    hit = {r["cluster"] for r in ranked}
    fig, ax = plt.subplots(figsize=(9, 7))
    bg = ~umap_df["cluster"].isin(hit)
    ax.scatter(umap_df.loc[bg, "x"], umap_df.loc[bg, "y"], s=5, c="#d9d9d9",
               alpha=0.5, linewidths=0)
    cmap = plt.cm.get_cmap("tab10", max(len(hit), 1))
    for i, cl in enumerate(sorted(hit)):
        m = umap_df["cluster"] == cl
        ax.scatter(umap_df.loc[m, "x"], umap_df.loc[m, "y"], s=18, color=cmap(i),
                   alpha=0.85, linewidths=0, label=f"Cluster {cl}")
    ax.set_title(f'Phenotype space - query: "{prompt}"', fontweight="bold")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2"); ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(STAGE3_DIR / "query_map.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_query_map("type 2 diabetes with renal complications")

In [ ]:
# Optional interactive Plotly version (hover shows note id + cluster)
try:
    import plotly.express as px
    ranked, _ = ps.retrieve("type 2 diabetes with renal complications")
    hit = {r["cluster"] for r in ranked}
    pdf = umap_df.copy()
    pdf["state"] = pdf["cluster"].apply(lambda c: f"Cluster {c}" if c in hit else "other")
    fig = px.scatter(pdf, x="x", y="y", color="state", hover_data=["note_id", "cluster"],
                     opacity=0.7, width=900, height=650,
                     title="PhenoPrompt - interactive query map")
    fig.update_traces(marker=dict(size=5))
    fig.write_html(str(STAGE3_DIR / "query_map_interactive.html"))
    fig.show()
    print("Saved stage3_outputs/query_map_interactive.html")
except Exception as e:
    print(f"plotly unavailable ({e}) - static map only.")

## 9. Level 3 - novel / under-labelled phenotype discovery

Because clusters are discovered without disease labels, some may not map cleanly onto a common
ICD category. We probe each cluster against a list of common conditions; clusters with weak
overlap are flagged as **candidate novel phenotypes** and the LLM proposes a hypothesis name
(framed as *recommend review*, never asserted as a diagnosis).

In [ ]:
COMMON_CONDITIONS = [
    "diabetes", "heart failure", "hypertension", "pneumonia", "copd", "asthma",
    "chronic kidney disease", "myocardial infarction", "stroke", "cancer", "sepsis",
]

def discover_novel():
    flagged = []
    for cl in ps.cluster_ids:
        mix = ps.phenotype_mix(cl)
        top_ents = mix["entity"].head(5).tolist()
        overlap = [e for e in top_ents
                   if any(c in e or e in c for c in COMMON_CONDITIONS)]
        if not overlap:                                   # no common disorder in the top mix
            label, narrative = synthesize(mix, ps.profiles[str(cl)]["n_notes"], top_ents, [])
            flagged.append(dict(cluster=cl, n_notes=ps.profiles[str(cl)]["n_notes"],
                                top_entities=top_ents, proposed_label=label,
                                hypothesis=narrative))
    return flagged

novel = discover_novel()
if novel:
    for n in novel:
        print(f"Cluster {n['cluster']} ({n['n_notes']} notes) - candidate novel phenotype")
        print(f"   top entities : {n['top_entities']}")
        print(f"   proposed name: {n['proposed_label']}")
        print(f"   hypothesis   : {textwrap.fill(n['hypothesis'], 90, subsequent_indent='   ')}\n")
else:
    print("No clusters flagged as novel - every cluster's top mix maps to a common condition.")

## 10. Under-coded conditions - notes vs structured codes (optional)

PhenoPrompt's most distinctive output: entities **frequent in notes but rare in structured
ICD/coded fields** - what clinicians wrote but coders missed. This needs a per-note table of
structured codes. Provide `stage1_outputs/structured_codes.csv` with columns
`note_id, condition` (one row per coded condition). If absent, this step is skipped.

> For `AGBonnet/augmented-clinical-notes` you can build this from the structured `summary`
> field (parse its diagnosis entries into condition surface forms) - dataset-specific, so it
> is left as a hook rather than hard-coded.

In [ ]:
CODES_CSV = STAGE1_DIR / "structured_codes.csv"

def undercoding_report():
    if not CODES_CSV.exists():
        print(f"{CODES_CSV} not found - skipping under-coding analysis.")
        print("Provide note_id,condition rows to enable it.")
        return None
    codes = pd.read_csv(CODES_CSV, dtype={"note_id": str})
    coded_sets = codes.groupby("note_id")["condition"].apply(lambda s: set(s.str.lower()))
    rows = []
    for cl in ps.cluster_ids:
        ids = ps.assign.loc[ps.assign.cluster == cl, "note_id"].values
        members = ps.count.reindex(ids).fillna(0)
        note_prev = (members > 0).mean()                       # documented in notes
        for ent in ps.phenotype_mix(cl, top=10)["entity"]:
            coded = np.mean([ent in coded_sets.get(i, set()) for i in ids]) if len(ids) else 0
            gap = note_prev.get(ent, 0) - coded
            if gap > 0.2:                                      # >=20-pt note-vs-code gap
                rows.append(dict(cluster=cl, entity=ent,
                                 note_prevalence=round(float(note_prev.get(ent, 0)), 3),
                                 coded_prevalence=round(float(coded), 3),
                                 gap=round(float(gap), 3)))
    out = pd.DataFrame(rows).sort_values("gap", ascending=False)
    out.to_csv(STAGE3_DIR / "undercoded_conditions.csv", index=False)
    print(f"Under-coded findings: {len(out)} "
          f"(saved to stage3_outputs/undercoded_conditions.csv)")
    return out

undercoded = undercoding_report()
if undercoded is not None and len(undercoded):
    print(undercoded.head(15).to_string(index=False))

## 11. Save Stage 3 outputs

In [ ]:
# NumPy types (int64/float64) sneak in via pandas; make json serialise them
def _json_default(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray):  return o.tolist()
    raise TypeError(f"{type(o)} not serializable")

import datetime

# 11.1 machine-readable query results (Level 1)
serialisable = {}
for q, reps in all_results.items():
    serialisable[q] = [dict(cluster=r["cluster"], label=r["label"], narrative=r["narrative"],
                            hybrid_score=r["hybrid_score"], n_notes=r["n_notes"],
                            matched_entities=r["matched_entities"],
                            phenotype_mix=r["phenotype_mix"].to_dict(orient="records"),
                            provenance=[{"note_id": f["note_id"], "spans": f["provenance"]}
                                        for f in r["fragments"]])
                       for r in reps]
(STAGE3_DIR / "query_results.json").write_text(json.dumps(serialisable, indent=2, default=_json_default))

# 11.2 a portable description of the queryable index (the phenotype "registry")
registry = dict(
    built=datetime.datetime.now().isoformat(timespec="seconds"),
    n_clusters=len(ps.cluster_ids), vocabulary=ps.vocab,
    retrieval=dict(alpha=ALPHA, top_k=TOP_K, method="entity-augmented + dense hybrid"),
    clusters={str(cl): ps.profiles[str(cl)] for cl in ps.cluster_ids},
)
(STAGE3_DIR / "phenospace_registry.json").write_text(json.dumps(registry, indent=2, default=_json_default))

print("Stage 3 outputs saved to", STAGE3_DIR)
for f in sorted(STAGE3_DIR.iterdir()):
    print(f"  {f.name:35s} {f.stat().st_size/1024:8.1f} KB")

## 12. Stage 3 summary

In [ ]:
print("=" * 60)
print("PHENOPROMPT - STAGE 3 SUMMARY")
print("=" * 60)
print(f"  Clusters indexed:        {len(ps.cluster_ids):>6}")
print(f"  Vocabulary size:         {len(ps.vocab):>6}")
print(f"  Note texts available:    {len(ps.note_texts):>6,}")
print(f"  Retrieval:               entity-augmented + dense hybrid (alpha={ALPHA})")
print(f"  LLM backend:             {LLM_BACKEND}")
print(f"  Example queries run:     {len(EXAMPLE_QUERIES):>6}")
print(f"  Novel-phenotype flags:   {len(novel):>6}")
print("")
print("Outputs:")
print("  query_results.json          - Level 1 per-query reports + provenance")
print("  phenospace_registry.json    - portable queryable phenotype registry")
print("  query_map.png               - Level 2 population map with query highlight")
print("  query_map_interactive.html  - interactive version (if plotly present)")
print("  undercoded_conditions.csv   - Level 3 note-vs-code gaps (if codes provided)")
print("=" * 60)
print("\nStage 3 complete - PhenoPrompt is now queryable end to end.")